In [36]:
"""
pctc_nexus_benchmark.py
=======================
Extracts logical metrics from the PCTC m=1 circuit,
compiles on H2-1E via Quantinuum Nexus, extracts compiled metrics,
and saves everything to pctc_metrics.json.

Usage:  python pctc_nexus_benchmark.py
"""

import time
import json
from collections import Counter
from math import pi

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.converters import circuit_to_dag
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.circuit import OpType
from pytket import Circuit as TKCircuit
import qnexus as qnx

# ── Config ────────────────────────────────────────────────────────────────────
PROJECT_NAME = "CTCs"
BACKEND      = "H2-1E"
OPT_LEVEL    = 3

# Fixed message parameters (same as used for DCTC for consistency)
THETA  = 2.5349076035276403
VARPHI = 2.0022404587009195

# Native 2q gate types for H2 family
TWO_Q_TYPES = {
    OpType.ZZMax, OpType.ZZPhase, OpType.PhasedISWAP,
    OpType.ISWAP, OpType.CX, OpType.CZ
}

# ── Build PCTC circuit ────────────────────────────────────────────────────────
def build_pctc() -> QuantumCircuit:
    # lowercase register names for QASM/pytket compatibility
    C = QuantumRegister(1, 'qc')
    E = QuantumRegister(1, 'qe')
    R = QuantumRegister(1, 'qr')
    G = QuantumRegister(1, 'qg')
    M = QuantumRegister(1, 'qm')
    A = QuantumRegister(1, 'qa')
    Y = QuantumRegister(1, 'qy')
    crR = ClassicalRegister(1, 'crr')
    crG = ClassicalRegister(1, 'crg')
    crC = ClassicalRegister(1, 'crc')

    qc = QuantumCircuit(C, E, R, G, M, A, Y, crR, crG, crC)

    # Message preparation
    qc.u(THETA, VARPHI, 0.0, M[0])

    # SWAP message into CTC
    qc.swap(C[0], M[0])
    qc.barrier()

    # EPR pairs
    qc.h(E[0]);  qc.cx(E[0], M[0])
    qc.h(R[0]);  qc.cx(R[0], G[0])
    qc.h(A[0]);  qc.cx(A[0], Y[0])
    qc.barrier()

    # Scrambling unitary
    qc.cz(C[0], R[0]); qc.cz(E[0], R[0]); qc.cz(C[0], E[0])
    qc.h(C[0]);  qc.h(E[0]);  qc.h(R[0])
    qc.cz(C[0], R[0]); qc.cz(C[0], E[0]); qc.cz(E[0], R[0])
    qc.barrier()

    # Probabilistic decoder — unitary conjugate
    qc.cz(A[0], G[0]); qc.cz(M[0], A[0]); qc.cz(G[0], M[0])
    qc.h(A[0]);  qc.h(M[0]);  qc.h(G[0])
    qc.cz(A[0], G[0]); qc.cz(G[0], M[0]); qc.cz(M[0], A[0])
    qc.barrier()

    # Bell projection
    qc.cx(R[0], G[0])
    qc.h(R[0])
    qc.measure(R[0], crR[0])
    qc.measure(G[0], crG[0])
    qc.barrier()

    # Swap back and uncompute message
    qc.swap(C[0], Y[0])
    qc.u(THETA, VARPHI, 0.0, C[0]).inverse()
    qc.measure(C[0], crC[0])

    return qc


# ── Logical metrics (Qiskit) ──────────────────────────────────────────────────
def logical_metrics(qc: QuantumCircuit) -> dict:
    dag = circuit_to_dag(qc)
    all_nodes = list(dag.topological_op_nodes())

    two_q = [n for n in all_nodes if len(n.qargs) == 2 and n.op.name != "measure"]
    one_q = [n for n in all_nodes if len(n.qargs) == 1 and n.op.name != "measure"]
    meas  = [n for n in all_nodes if n.op.name == "measure"]

    n2q   = len(two_q)
    n1q   = len(one_q)
    nmeas = len(meas)
    ntot  = n2q + n1q   # measurements excluded

    d = qc.depth()

    # 2q-only depth
    sub = qc.copy_empty_like()
    for node in dag.topological_op_nodes():
        if len(node.qargs) == 2 and node.op.name != "measure":
            sub.append(node.op, node.qargs)
    d_2q = circuit_to_dag(sub).depth()

    gate_inventory = dict(Counter(
        n.op.name for n in all_nodes if n.op.name not in ("barrier", "measure")
    ))

    return {
        "circuit":         "PCTC",
        "m":               1,
        "backend":         BACKEND,
        "N_2q_logical":    n2q,
        "N_1q_logical":    n1q,
        "N_meas_logical":  nmeas,
        "N_gates_total_logical": ntot,
        "d_logical":       d,
        "d_2q_logical":    d_2q,
        "gate_inventory_logical": gate_inventory,
    }


# ── Compiled metrics (pytket) ─────────────────────────────────────────────────
def compiled_metrics(compiled_tk) -> dict:
    cmds = compiled_tk.get_commands()

    n2q   = sum(1 for c in cmds if c.op.type in TWO_Q_TYPES)
    n1q   = sum(1 for c in cmds if len(c.args) == 1
                                and c.op.type != OpType.Measure
                                and c.op.type != OpType.Reset)
    nmeas = sum(1 for c in cmds if c.op.type == OpType.Measure)
    ntot  = n2q + n1q   # measurements excluded

    d = compiled_tk.depth()

    # 2q-only depth
    sub = TKCircuit(compiled_tk.n_qubits, compiled_tk.n_bits)
    for c in cmds:
        if c.op.type in TWO_Q_TYPES:
            sub.add_gate(c.op.type, c.op.params, c.args)
    d_2q = sub.depth()

    gate_inventory = dict(Counter(
        str(c.op.type) for c in cmds
        if c.op.type not in (OpType.Barrier, OpType.Measure)
    ))

    return {
        "n_qubits":        compiled_tk.n_qubits,
        "N_2q_compiled":   n2q,
        "N_1q_compiled":   n1q,
        "N_meas":          nmeas,
        "N_gates_total":   ntot,
        "d_compiled":      d,
        "d_2q_compiled":   d_2q,
        "gate_inventory":  gate_inventory,
    }


# ── Main ──────────────────────────────────────────────────────────────────────
def run():
    print("=" * 55)
    print("  PCTC  m=1  →  H2-1E")
    print("=" * 55)

    # ── 1. Build circuit ─────────────────────────────────────────────────────
    qc = build_pctc()
    print(f"\nQiskit circuit: {qc.num_qubits}q / {qc.num_clbits} bits")

    # ── 2. Logical metrics ───────────────────────────────────────────────────
    log = logical_metrics(qc)
    print(f"\nLogical metrics:")
    print(f"  N_2q  = {log['N_2q_logical']}")
    print(f"  N_1q  = {log['N_1q_logical']}")
    print(f"  N_tot = {log['N_gates_total_logical']}  (excl. measurements)")
    print(f"  d     = {log['d_logical']}")
    print(f"  d_2q  = {log['d_2q_logical']}")
    print(f"  Gate inventory: {log['gate_inventory_logical']}")

    # ── 3. Convert to pytket ─────────────────────────────────────────────────
    tk_circ = qiskit_to_tk(qc)
    print(f"\npytket circuit: {tk_circ.n_qubits}q / {tk_circ.n_bits} bits")

    # ── 4. Connect to Nexus & upload ─────────────────────────────────────────
    project = qnx.projects.get(name=PROJECT_NAME)
    print(f"Project: {project.annotations.name}")

    circuit_ref = qnx.circuits.upload(
        name="PCTC_m1_benchmark",
        circuit=tk_circ,
        project=project,
    )
    print(f"Uploaded: {circuit_ref.annotations.name}")

    # ── 5. Compile on H2-1E ──────────────────────────────────────────────────
    compile_job = qnx.start_compile_job(
        programs=circuit_ref,
        backend_config=qnx.QuantinuumConfig(device_name=BACKEND),
        optimisation_level=OPT_LEVEL,
        name="PCTC_m1_compile",
        project=project,
    )
    print(f"Compile job submitted: {compile_job.id}")

    # ── 6. Poll ───────────────────────────────────────────────────────────────
    print("Waiting", end="", flush=True)
    while True:
        status = qnx.jobs.status(compile_job)
        if status.status.value in ("COMPLETED", "ERROR", "CANCELLED"):
            break
        print(".", end="", flush=True)
        time.sleep(5)
    print(f" {status.status.value}")

    if status.status.value != "COMPLETED":
        raise RuntimeError(f"Compile job failed: {status}")

    # ── 7. Download compiled circuit ─────────────────────────────────────────
    compiled_tk = qnx.jobs.results(compile_job)[0].get_output().download_circuit()
    print(f"Downloaded compiled circuit: {compiled_tk.n_qubits}q / {compiled_tk.n_bits} bits")

    # ── 8. Compiled metrics ──────────────────────────────────────────────────
    comp = compiled_metrics(compiled_tk)
    print(f"\nCompiled metrics ({BACKEND}):")
    print(f"  N_2q  = {comp['N_2q_compiled']}")
    print(f"  N_1q  = {comp['N_1q_compiled']}")
    print(f"  N_tot = {comp['N_gates_total']}  (excl. measurements)")
    print(f"  d     = {comp['d_compiled']}")
    print(f"  d_2q  = {comp['d_2q_compiled']}")
    print(f"  Gate inventory: {comp['gate_inventory']}")

    # ── 9. Save ───────────────────────────────────────────────────────────────
    result = {**log, **comp}
    with open("pctc_metrics.json", "w") as f:
        json.dump(result, f, indent=2)
    print(f"\nSaved → pctc_metrics.json")

    return result


if __name__ == "__main__":
    run()

  PCTC  m=1  →  H2-1E

Qiskit circuit: 7q / 3 bits

Logical metrics:
  N_2q  = 18
  N_1q  = 12
  N_tot = 30  (excl. measurements)
  d     = 24
  d_2q  = 8
  Gate inventory: {'u': 2, 'swap': 2, 'h': 10, 'cx': 4, 'cz': 12}

pytket circuit: 7q / 3 bits
Project: CTCs
Uploaded: PCTC_m1_benchmark
Compile job submitted: 5b7f8daa-a855-4827-9ed3-1587f80eeadd
Waiting.. COMPLETED
Downloaded compiled circuit: 7q / 3 bits

Compiled metrics (H2-1E):
  N_2q  = 12
  N_1q  = 29
  N_tot = 41  (excl. measurements)
  d     = 18
  d_2q  = 8
  Gate inventory: {'OpType.PhasedX': 27, 'OpType.ZZPhase': 12, 'OpType.Rz': 2}

Saved → pctc_metrics.json


In [7]:
import qnexus as qnx

backends = qnx.devices.get_all()
h2_row = backends[backends['device_name'] == 'H2-1E'].iloc[0]
info = h2_row['backend_info']

print(type(info))
print(dir(info))
print(info)

TypeError: list indices must be integers or slices, not str

In [34]:
import qnexus as qnx

backends = qnx.devices.get_all()
print(type(backends))
print(backends[13])

<class 'qnexus.models.references.DataframableList'>
backend_name='Quantinuum' device_name='H2-1' nexus_hosted=False stored_backend_info=StoredBackendInfo(name='EmulatorEnabledQuantinuumBackend', device_name='H2-1', version='0.55.1', device=StoredDevice(nodes=[], edges=[], n_nodes=56, fully_connected=True), gate_set=['Measure', 'Reset', 'PhasedX', 'Barrier', 'ZZMax', 'ZZPhase', 'WASM', 'SetBits', 'CopyBits', 'RangePredicate', 'ExplicitPredicate', 'ExplicitModifier', 'MultiBit', 'Rz', 'TK2', 'ClExpr', 'RNGSeed', 'RNGBound', 'RNGIndex', 'RNGNum', 'JobShotNum'], n_cl_reg=4000, supports_fast_feedforward=True, supports_reset=True, supports_midcircuit_measurement=True, misc={'wasm': True, 'batching': True, 'supported_languages': ['OPENQASM 2.0', 'QIR 1.0'], 'benchmarks': {'qv': {'date': '2024-08-11', 'value': 2097152.0}}, 'max_classical_register_width': 63, 'syntax_checker': 'H2-1SC', 'n_gate_zones': '4', 'noise_specs': {'date': '2025-04-30', 'spam_error': {'p_meas_1_unc': 0.000124, 'p_meas_0

In [37]:
import qnexus as qnx

backends = qnx.devices.get_all()

# Find H2-1 (which has H2-1E emulator)
for b in backends:
    if b.device_name == 'H2-1':
        noise = b.stored_backend_info.misc['noise_specs']
        print(f"Date: {noise['date']}")
        print(f"1q gate error:  p1  = {noise['1q_gate_error']['p1']:.2e}  ± {noise['1q_gate_error']['p1_unc']:.2e}")
        print(f"2q gate error:  p2  = {noise['2q_gate_error']['p2']:.2e}  ± {noise['2q_gate_error']['p2_unc']:.2e}")
        print(f"Readout error:  p(1|0) = {noise['spam_error']['p_meas_0']:.2e}")
        print(f"                p(0|1) = {noise['spam_error']['p_meas_1']:.2e}")
        print(f"Memory error:   {noise['memory_error']['memory_error']:.2e}")
        break

Date: 2025-04-30
1q gate error:  p1  = 1.89e-05  ± 4.23e-06
2q gate error:  p2  = 1.05e-03  ± 8.08e-05
Readout error:  p(1|0) = 6.00e-04
                p(0|1) = 1.39e-03
Memory error:   2.03e-04
